In [1]:
import os
from collections import defaultdict

import numpy as np
from coffea.util import load
import hist

import rich
import correctionlib
import correctionlib.convert

In [2]:
filename = "/work/mmarcheg/BTVNanoCommissioning/output/test/pt_reweighting/pt_eta_tau21_reweighting_2018_twojets_pt350_leadsublead_v12/output_all.coffea"
output_dir = "maps"
pt_eta_2d_maps = []
pt_eta_tau21_3d_maps = ['FatJetGoodNMuon1_pt_eta_tau21_bintau05']
year = "2018"
histname = 'FatJetGoodNMuon1_pt_eta_tau21_bintau05'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
o = load(filename)
o.keys()

dict_keys(['sum_genweights', 'sumw', 'cutflow', 'variables', 'columns', 'processing_metadata', 'seed_fatjet_chunk'])

In [11]:
def dense_axes(h):
    '''Returns the list of dense axes of a histogram.'''
    dense_axes = []
    if type(h) == dict:
        h = h[list(h.keys())[0]]
    for ax in h.axes:
        if not type(ax) in [hist.axis.StrCategory, hist.axis.IntCategory]:
            dense_axes.append(ax)
    return dense_axes

def stack_sum(stack):
    '''Returns the sum histogram of a stack (`hist.stack.Stack`) of histograms.'''
    if len(stack) == 1:
        return stack[0]
    else:
        htot = stack[0]
        for h in stack[1:]:
            htot = htot + h
        return htot

def get_axis_items(h, axis_name):
    axis = h.axes[axis_name]
    return list(axis.value(range(axis.size)))

def get_data_mc_ratio(h_data, h_qcd, h_diff):
    if type(h_data) == hist.Stack:
        h_data = stack_sum(h_data)
    if type(h_qcd) == hist.Stack:
        h_qcd = stack_sum(h_qcd)
    if type(h_diff) == hist.Stack:
        h_diff = stack_sum(h_diff)
    data = h_data.values()
    qcd = h_qcd.values()
    diff = h_diff.values()
    num = data - diff
    den = qcd
    ratio = num / den
    sumw2_num = h_data.variances() + h_diff.variances()
    sumw2_den = h_qcd.variances()
    # Statistical uncertainty on the reweighting SF taking into account
    # the uncertainty on data, QCD, top and WJets
    unc = np.sqrt( sumw2_num/den**2 + (num**2/den**4)*sumw2_den )
    # Statistical uncertainty on the reweighting SF taking into account
    # the uncertainty on data and QCD only
    unc_no_diff = np.sqrt( data/den**2 + (num**2/den**4)*sumw2_den )
    unc[np.isnan(unc)] = np.inf
    unc_no_diff[np.isnan(unc_no_diff)] = np.inf

    return ratio, unc, unc_no_diff

def overwrite_check(outfile):
    path = outfile
    version = 1
    while os.path.exists(path):
        tag = str(version).rjust(2, '0')
        path = outfile.replace('.json', f'_v{tag}.json')
        version += 1
    if path != outfile:
        print(f"The output will be saved to {path}")
    return path

def pt_reweighting(accumulator, year, histname):
    h = accumulator['variables'][histname]
    samples = h.keys()
    samples_data = list(filter(lambda d: 'DATA' in d, samples))
    samples_mc = list(filter(lambda d: 'DATA' not in d, samples))
    samples_qcd = list(filter(lambda d: 'QCD' in d, samples_mc))
    samples_vjets_top = list(filter(lambda d: (('VJets' in d) | ('SingleTop_ttbar' in d)), samples_mc))

    h_qcd = h[samples_qcd[0]]

    axes = dense_axes(h_qcd)
    categories = get_axis_items(h_qcd, 'cat')

    ratio_dict = defaultdict(float)

    for cat in categories:
        ratio_dict[cat] = {}
        for var_shape in ["nominal", "JES_TotalUp", "JES_TotalDown", "JERUp", "JERDown"]:
            slicing_mc = {'year': year, 'cat': cat, 'variation': var_shape}
            dict_qcd = {d: h[d][slicing_mc] for d in samples_qcd}
            dict_vjets_top = {d: h[d][slicing_mc] for d in samples_vjets_top}
            stack_qcd = hist.Stack.from_dict(dict_qcd)
            stack_vjets_top = hist.Stack.from_dict(dict_vjets_top)

            if 'era' in h[samples_data[0]].axes.name:
                slicing_data = {'year': year, 'cat': cat, 'era': sum}
            else:
                slicing_data = {'year': year, 'cat': cat}
            dict_data = {d: h[d][slicing_data] for d in samples_data}
            stack_data = hist.Stack.from_dict(dict_data)
            if len(stack_data) > 1:
                raise NotImplementedError
            ratio, unc, unc_no_diff = get_data_mc_ratio(stack_data, stack_qcd, stack_vjets_top)
            mod_ratio  = np.nan_to_num(ratio)
            mod_unc = np.nan_to_num(unc)
            mod_unc_no_diff = np.nan_to_num(unc_no_diff)
            #if histname in self.pt_eta_2d_maps:
            #    mod_ratio[mod_ratio == 0.0] = 1

            ratio_dict[cat][var_shape] = {}
            ratio_dict[cat][var_shape].update({ "nominal" : mod_ratio })
            ratio_dict[cat][var_shape].update({ "statUp" : mod_ratio + mod_unc })
            ratio_dict[cat][var_shape].update({ "statDown" : mod_ratio - mod_unc })

    categories = list(ratio_dict.keys())
    shape_variations = list(ratio_dict[categories[0]].keys())
    variations = list(ratio_dict[categories[0]][shape_variations[0]].keys())
    axis_category = hist.axis.StrCategory(categories, name="cat")
    axis_shape_variation = hist.axis.StrCategory(shape_variations, name="shape")
    axis_variation = hist.axis.StrCategory(variations, name="variation")
    # Stack nominal, statUp, statDown maps for each category
    stack_map = np.stack([[list(ratio_dict[cat][var_shape].values()) for var_shape in shape_variations] for cat in categories])
    sfhist = hist.Hist(axis_category, axis_shape_variation, axis_variation, *axes, data=stack_map)
    sfhist.label = "out"
    sfhist.name = f"{histname}_corr_{year}"
    description = "Reweighting SF matching the leading fatjet pT and eta MC distribution to data."
    clibcorr = correctionlib.convert.from_histogram(sfhist, flow="clamp")
    clibcorr.description = description
    cset = correctionlib.schemav2.CorrectionSet(
        schema_version=2,
        description="MC to data reweighting SF",
        corrections=[clibcorr],
    )
    rich.print(cset)

    outfile_reweighting = os.path.join(output_dir, f'{histname}_{year}_reweighting.json')
    outfile_reweighting = overwrite_check(outfile_reweighting)
    print(f"Saving pt reweighting factors in {outfile_reweighting}")
    with open(outfile_reweighting, "w") as fout:
        fout.write(cset.json(exclude_unset=True))
    fout.close()
    print(f"Loading correction from {outfile_reweighting}...")
    cset = correctionlib.CorrectionSet.from_file(outfile_reweighting)
    print("(cat): inclusive,", "(var): nominal")
    pt_corr = cset[sfhist.name]
    pos  = np.array([0, 1, 0, 1, 0], dtype=int)
    pt  = np.array([50, 100, 400, 500, 1000], dtype=float)
    eta = np.array([-2, -1, 0, 1, 2], dtype=float)
    print("pos =", pos)
    print("pt =", pt)
    print("eta =", eta)
    if histname in pt_eta_2d_maps:
        args = (pos, pt, eta)

    elif histname in pt_eta_tau21_3d_maps:
        tau21 = np.array([0.1, 0.35, 0.65, 0.8, 0.9], dtype=float)
        print("tau21 =", tau21)
        args = (pos, pt, eta, tau21)
    for var_shape in shape_variations:
        categorical_args = ['inclusive', var_shape, 'nominal']
        print(categorical_args)
        print(pt_corr.evaluate(*categorical_args, *args))

In [12]:
pt_reweighting(o, year, histname)

/tmp/ipykernel_28186/1330200055.py:37: RuntimeWarning: divide by zero encountered in divide
  ratio = num / den
/tmp/ipykernel_28186/1330200055.py:37: RuntimeWarning: invalid value encountered in divide
  ratio = num / den
/tmp/ipykernel_28186/1330200055.py:42: RuntimeWarning: divide by zero encountered in divide
  unc = np.sqrt( sumw2_num/den**2 + (num**2/den**4)*sumw2_den )
/tmp/ipykernel_28186/1330200055.py:42: RuntimeWarning: invalid value encountered in divide
  unc = np.sqrt( sumw2_num/den**2 + (num**2/den**4)*sumw2_den )
/tmp/ipykernel_28186/1330200055.py:42: RuntimeWarning: invalid value encountered in multiply
  unc = np.sqrt( sumw2_num/den**2 + (num**2/den**4)*sumw2_den )
/tmp/ipykernel_28186/1330200055.py:45: RuntimeWarning: divide by zero encountered in divide
  unc_no_diff = np.sqrt( data/den**2 + (num**2/den**4)*sumw2_den )
/tmp/ipykernel_28186/1330200055.py:45: RuntimeWarning: invalid value encountered in divide
  unc_no_diff = np.sqrt( data/den**2 + (num**2/den**4)*sumw

CorrectionSet (schema v2)
MC to data reweighting SF
📂
└── 📈 FatJetGoodNMuon1_pt_eta_tau21_bintau05_corr_2018 (v0)
    Reweighting SF matching the leading fatjet pT and eta MC distribution to data.
    Node counts: Category: 22, MultiBinning: 30
    ╭────────────── ▶ input ──────────────╮ ╭────────────────────────── ▶ input ──────────────────────────╮
    │ cat (string)                        │ │ shape (string)                                              │
    │ cat                                 │ │ shape                                                       │
    │ Values: inclusive                   │ │ Values: JERDown, JERUp, JES_TotalDown, JES_TotalUp, nominal │
    ╰─────────────────────────────────────╯ ╰─────────────────────────────────────────────────────────────╯
    ╭────────────── ▶ input ──────────────╮ ╭────────────────────────── ▶ input ──────────────────────────╮
    │ variation (string)                  │ │ FatJetGoodNMuon1_pos (int)                                  │
    │ variation                           │ │ FatJet position                                             │
    │ Values: nominal, statDown, statUp   │ │ Values: 0, 1                                                │
    ╰─────────────────────────────────────╯ ╰─────────────────────────────────────────────────────────────╯
    ╭────────────── ▶ input ──────────────╮ ╭────────────────────────── ▶ input ──────────────────────────╮
    │ FatJetGoodNMuon1_pt (real)          │ │ FatJetGoodNMuon1_eta (real)                                 │
    │ FatJet $p_{T}$ [GeV]                │ │ FatJet $\eta$                                               │
    │ Range: [350.0, 2500.0), overflow ok │ │ Range: [-5.0, 5.0), overflow ok                             │
    ╰─────────────────────────────────────╯ ╰─────────────────────────────────────────────────────────────╯
    ╭────────────── ▶ input ──────────────╮                                                                
    │ FatJetGoodNMuon1_tau21 (real)       │                                                                
    │ FatJet $\tau_{21}$                  │                                                                
    │ Range: [0.0, 1.0), overflow ok      │                                                                
    ╰─────────────────────────────────────╯                                                                
    ╭─── ◀ output ───╮
    │ out (real)     │
    │ No description │
    ╰────────────────╯

The output will be saved to maps/FatJetGoodNMuon1_pt_eta_tau21_bintau05_2018_reweighting_v04.json
Saving pt reweighting factors in maps/FatJetGoodNMuon1_pt_eta_tau21_bintau05_2018_reweighting_v04.json
Loading correction from maps/FatJetGoodNMuon1_pt_eta_tau21_bintau05_2018_reweighting_v04.json...
(cat): inclusive, (var): nominal
pos = [0 1 0 1 0]
pt = [  50.  100.  400.  500. 1000.]
eta = [-2. -1.  0.  1.  2.]
tau21 = [0.1  0.35 0.65 0.8  0.9 ]
['inclusive', 'nominal', 'nominal']
[-1.25046251  0.31122235  2.34829942  0.          0.        ]
['inclusive', 'JES_TotalUp', 'nominal']
[-1.4008789   0.31122235  2.33087698  0.          0.        ]
['inclusive', 'JES_TotalDown', 'nominal']
[-1.79769313e+308  1.48855988e+001  2.29282095e+000  0.00000000e+000
  0.00000000e+000]
['inclusive', 'JERUp', 'nominal']
[-1.38169495  0.31122235  2.34829942  0.          0.        ]
['inclusive', 'JERDown', 'nominal']
[-1.25046251  0.29791973  2.2614518   0.          0.        ]


In [10]:
outfile_reweighting = "maps/FatJetGoodNMuon1_pt_eta_tau21_bintau05_2018_reweighting.json"
print(f"Loading correction from {outfile_reweighting}...")
cset = correctionlib.CorrectionSet.from_file(outfile_reweighting)
print("(cat): inclusive,", "(var): nominal")

Loading correction from maps/FatJetGoodNMuon1_pt_eta_tau21_bintau05_2018_reweighting.json...
(cat): inclusive, (var): nominal


In [16]:
pt_corr = cset["FatJetGoodNMuon1_pt_eta_tau21_bintau05_corr_2018"]
pos  = np.array([0, 1, 0, 1, 0], dtype=int)
pt  = np.array([50, 100, 400, 500, 1000], dtype=float)
eta = np.array([-2, -1, 0, 1, 2], dtype=float)
print("pos =", pos)
print("pt =", pt)
print("eta =", eta)
if histname in pt_eta_2d_maps:
    args = (pos, pt, eta)

elif histname in pt_eta_tau21_3d_maps:
    tau21 = np.array([0.1, 0.35, 0.65, 0.8, 0.9], dtype=float)
    print("tau21 =", tau21)
    args = (pos, pt, eta, tau21)
print(pt_corr.evaluate('inclusive', 'nominal', 'nominal', *args))
print(pt_corr.evaluate('inclusive', 'JES_TotalUp', 'nominal', *args))
print(pt_corr.evaluate('inclusive', 'JES_TotalDown', 'nominal', *args))

pos = [0 1 0 1 0]
pt = [  50.  100.  400.  500. 1000.]
eta = [-2. -1.  0.  1.  2.]
tau21 = [0.1  0.35 0.65 0.8  0.9 ]


IndexError: Index not available in Category for input argument 1 val: nominal

In [15]:
filename = "/work/mmarcheg/BTVNanoCommissioning/output/test/pt_reweighting/pt_eta_tau21_reweighting_2018_twojets_pt350_leadsublead_v16/FatJetGoodNMuon1_pt_eta_tau21_bintau05_2018_reweighting.json"
cset = correctionlib.CorrectionSet.from_file(filename)
pt_corr = cset["FatJetGoodNMuon1_pt_eta_tau21_bintau05_corr_2018"]
pos  = np.array([0, 1, 0, 1, 0], dtype=int)
pt  = np.array([50, 100, 400, 500, 1000], dtype=float)
eta = np.array([-2, -1, 0, 1, 2], dtype=float)
print("pos =", pos)
print("pt =", pt)
print("eta =", eta)
if histname in pt_eta_2d_maps:
    args = (pos, pt, eta)

elif histname in pt_eta_tau21_3d_maps:
    tau21 = np.array([0.1, 0.35, 0.65, 0.8, 0.9], dtype=float)
    print("tau21 =", tau21)
    args = (pos, pt, eta, tau21)
print(pt_corr.evaluate('inclusive', 'nominal', 'nominal', *args))
print(pt_corr.evaluate('inclusive', 'JES_TotalUp', 'nominal', *args))
print(pt_corr.evaluate('inclusive', 'JES_TotalDown', 'nominal', *args))

pos = [0 1 0 1 0]
pt = [  50.  100.  400.  500. 1000.]
eta = [-2. -1.  0.  1.  2.]
tau21 = [0.1  0.35 0.65 0.8  0.9 ]
[-1.25046251  0.31122235  2.34829942  0.          0.        ]
[-1.4008789   0.31122235  2.33087698  0.          0.        ]
[-1.79769313e+308  1.48855988e+001  2.29282095e+000  0.00000000e+000
  0.00000000e+000]


In [21]:
dir(pt_corr)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_base',
 '_context',
 '_name',
 'description',
 'evaluate',
 'inputs',
 'name',
 'output',
 'version']

In [28]:
for i in pt_corr.inputs:
    print(i.name)

cat
shape
variation
FatJetGoodNMuon1_pos
FatJetGoodNMuon1_pt
FatJetGoodNMuon1_eta
FatJetGoodNMuon1_tau21
